In [ ]:
!pip install -U langchain langchain-google-genai langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found exis

In [ ]:
!pip install -U langchain langchain-google-genai langchain-community

In [ ]:
import sqlite3
import os

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [ ]:
DB_NAME = "students.db"

# Connect to SQLite
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
""")

# Insert student data
students = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88)
]

cursor.executemany("""
INSERT OR REPLACE INTO students
(student_id, name, department, python, database, ai, web)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", students)

conn.commit()
conn.close()

print("students.db created successfully!")

students.db created successfully!


In [ ]:
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("SELECT * FROM students")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78)
('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72)
('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90)
('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62)
('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88)


In [ ]:
@tool
def get_student_info(student_id: str) -> str:
    """
    Get the student's name and department using their student ID.
    Use this tool when the user asks for a student's name or department.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT name, department
        FROM students
        WHERE student_id = ?
    """, (student_id,))

    result = cursor.fetchone()
    conn.close()

    if result is None:
        return f"No student found with ID {student_id}."

    name, department = result

    return (
        f"Student ID: {student_id}\n"
        f"Name: {name}\n"
        f"Department: {department}"
    )

In [ ]:
@tool
def get_student_marks(student_id: str) -> str:
    """
    Get the Python, Database, AI, and Web marks of a student.
    Use this tool when the user asks about marks, total, average, or passing eligibility.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT python, database, ai, web
        FROM students
        WHERE student_id = ?
    """, (student_id,))

    result = cursor.fetchone()
    conn.close()

    if result is None:
        return f"No student found with ID {student_id}."

    python_mark, database_mark, ai_mark, web_mark = result

    return (
        f"Student ID: {student_id}\n"
        f"Python: {python_mark}\n"
        f"Database: {database_mark}\n"
        f"AI: {ai_mark}\n"
        f"Web: {web_mark}"
    )

In [ ]:
@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    Use this tool to calculate totals and averages from student marks.
    Examples:
    85 + 72 + 90 + 78
    (85 + 72 + 90 + 78) / 4
    """

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"

    except Exception as e:
        return f"Could not calculate the expression: {e}"

In [ ]:
@tool
def get_passing_rules() -> str:
    """
    Get the university rules for passing.
    Use this tool when determining whether a student satisfies the passing requirements.
    """

    return """
University Passing Rules:

1. Minimum overall average: 40%
2. Minimum mark in each subject: 35%

A student passes only if:
- Their average is at least 40%
- Every subject mark is at least 35%
"""

In [ ]:
tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules
]

print("Tools available to the agent:")

for tool in tools:
    print("-", tool.name)

Tools available to the agent:
- get_student_info
- get_student_marks
- calculator
- get_passing_rules


In [ ]:
os.environ["GOOGLE_API_KEY"] = "Google_API_Key"

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini model created successfully!")

Gemini model created successfully!


In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a student information assistant.

You have access to tools that can retrieve student information,
student marks, calculate mathematical expressions, and retrieve
university passing rules.

Always use the appropriate tools to answer questions.

Do not guess student information or marks.

When calculating total or average marks, use the calculator tool.

When checking passing eligibility:
1. Get the student's marks.
2. Get the university passing rules.
3. Calculate the total and average using the calculator when required.
4. Compare the student's marks and average with the rules.
5. Give a clear final answer.

You decide which tools are needed based on the user's question.
Do not follow a fixed tool sequence unless the question requires it.
"""
)

print("Agent created successfully!")

Agent created successfully!


In [ ]:
def ask_agent(question):
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    return response

In [ ]:
response = ask_agent(
    "What is the name and department of student 22CS045?"
)

print(response["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The name of student 22CS045 is **Dhanushya** and her department is **Computer Science**.', 'extras': {'signature': 'EucBCuQBAWkUfRN65I7CiEhTSJ+wZCQ4Q5eAdwbLcNT67ACfTCZPLlVX3wEpqFwHpWCb7Eqqa5wKy8p3ywQgj80yNJyqqWXF/1ZGgOmrPfH5HzexfyAI1WWObRl8cqYRpb5SZT6PhFUPxsgLovfZuBaNKZ7pfiAKAH61JOkg5NUQ8tRBObanixQDhq+TAmnqGX+g3M8OvYWqdyQTTeioZjQNGvha8eROuoCcnQZ6bjqI5stkQy2d7Nq+r7ucW09zO51qynEM9tFSEGjgRtsTQDpNjx9cn4yKKFPimNXugfFfRx58XXYu01v+'}}]


In [ ]:
response = ask_agent(
    "What are the marks of 22CS047?"
)

print(response["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The marks for student **22CS047** are as follows:\n\n* **Python:** 92\n* **Database:** 88\n* **AI:** 95\n* **Web:** 90', 'extras': {'signature': 'EoQCCoECAWkUfRMm/B/0cDlmp02R6AWsKyXMI1mip+ryRIjWinDZ8dq+xMELq1ifVItbQ0foIijDhbKdSHlRYAa5vkDw8mMdbWsMojYveLJ2LFZkZxwpB6VAEC/MqLsjqNd1rCOET07f9TJb5sVKS4BOLrVNcfQgFbu3ENvt4VEDQf2s+9qfnlSXWQv8Yf9M5NYE+5JNNFRx1z9LG4O4kVhmtONeV+OSIy9uMHS8NxfrFd5HNHO1LANCLY+uLa3YMf4ZaAqEO4lbCUtRR1QoTpfD1kE0544Gc32DTKV9pbTVz1qO2M0NZijse5AgF9wc6i/vLByyFNo5AS0VgOMIsS607T9KJSI='}}]


In [ ]:
response = ask_agent(
    "What is the total and average mark of 22CS045?"
)

print(response["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'For student **22CS045**, the marks obtained are:\n- **Python:** 85\n- **Database:** 72\n- **AI:** 90\n- **Web:** 78\n\n**Summary:**\n- **Total Marks:** 325\n- **Average Mark:** 81.25', 'extras': {'signature': 'EscBCsQBAWkUfRN22sd3+zmziQi3lSG5JkicQicjM9VJp8eFkqH40gh7xvjAvi3+YYMmA//Zr/AoejXJ0rWBfxMBSf1ccNcH58UTkUwusivnaA8WUAke6to98/HIXNtCBTlEJ1M2aMQd7jHviZFYjLCkRPfzr3g86TiZDon1IM04plrikk9JqoY8wzrvFYbxDQNJYZqoMlh9Qh1+/FulrODNZiW8VLZhU9SuSlmXZ9R990qXi7/feEW8CoRyzFwrVVFJb1rR3fBzwg=='}}]


In [ ]:
response = ask_agent(
    "Is 22CS045 eligible to pass according to the university rules?"
)

print(response["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "Based on the university passing rules and the student's performance, **22CS045 is eligible to pass**.\n\n---\n\n### **1. Student Marks**\n* **Python:** 85\n* **Database:** 72\n* **AI:** 90\n* **Web:** 78\n\n---\n\n### **2. Performance Analysis**\n* **Total Marks:** 325 / 400\n* **Overall Average:** 81.25%\n\n---\n\n### **3. Evaluation Against University Rules**\n1. **Subject Requirement:** Minimum **35%** in each subject.\n   * *Status:* **PASS** (Lowest mark is 72 in Database, which is above 35%).\n2. **Overall Average Requirement:** Minimum **40%** overall average.\n   * *Status:* **PASS** (Student achieved 81.25%, which is above 40%).\n\n---\n\n### **Final Verdict**\nStudent **22CS045 satisfies all passing requirements** and is **eligible to pass**.", 'extras': {'signature': 'EpsECpgEAWkUfRMjSG0PzR/DYwZ5+dV1M96WPkpI03sL4cXHy3YMvk8pQvPf03HGVw8aCGWZRnUUtHfELTlN7hVSpW3XCsvSCmyuin7oCue80TCNiEZ9INBSDw2BevtKb6QK0gaUoY0IvN0zF15Eq54NkBLHUZoi05U5FT8s9gEDNkK1eQD2jrK